# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a complete walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows the MLCommons Croissant schema.

Dataset identifier: `10.71728/senscience.qs2f-h81p`

[View dataset on sen.science](https://sen.science/doi/10.71728/senscience.qs2f-h81p/)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema URL using `mlcroissant`.

We load the full schema and print key information as a quick description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Let's review the available record sets and their structure.

### Available Record Sets
All references use the entities' `@id` as required by the Croissant schema.


In [ ]:
# List available RecordSets, their @id and fields
record_sets = list(ds.record_sets)
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    if 'name' in rs:
        print(f"  Name: {rs['name']}")
    if 'description' in rs:
        print(f"  Description: {rs['description']}")
    field_ids = []
    if 'field' in rs:
        # field could be list of dicts (fields), or a single dict
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                field_ids.append(f["@id"])
            elif isinstance(f, str):
                # Sometimes fields are given as direct @id strings
                field_ids.append(f)
    if field_ids:
        print("  Fields (@id):", ", ".join(field_ids))
    print()

For each record set, let’s preview the field names and types. We will also preview a few rows per record set.

In [ ]:
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nRecords for RecordSet @id: {rs_id}")
    # Print schema field names and types, if defined
    if 'field' in rs and isinstance(rs['field'], list):
        print("  Fields and data types:")
        for f in rs['field']:
            if isinstance(f, dict):
                fid = f.get('@id','')
                fname = f.get('name','')
                ftype = f.get('dataType','')
                print(f"    {fid} | name: {fname} | type: {ftype}")
            elif isinstance(f, str):
                print(f"    {f}")
    # Preview first 2 records
    try:
        records = list(ds.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            print(df.head(2))
        else:
            print("  [EMPTY]")
    except Exception as e:
        print(f"  Could not load records: {e}")

## 3. Data Extraction
We will extract all record sets into individual Pandas DataFrames using their record set `@id`.

This allows us to select and work with any record set by its `@id` in all following steps.

In [ ]:
# Extract all record sets by @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(ds.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
    except Exception as e:
        print(f"Warning: Could not load DataFrame for {rs_id}: {e}")

# For demonstration, pick the largest dataframe loaded (usually the main tabular dataset)
main_rs_id = None
max_rows = 0
for k, df in dataframes.items():
    if df.shape[0] > max_rows:
        main_rs_id = k
        max_rows = df.shape[0]

print(f"Main data RecordSet chosen: {main_rs_id}")
if main_rs_id:
    print("Available columns/@ids:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process and explore the data:

* Filter rows based on a numeric field (e.g., Age)
* Normalize the field
* Group by a categorical field (e.g., Sex or Cancer Type)

> **Note**: Please replace `<numeric_field_id>` and `<group_field_id>` below with the actual field `@id` from above (e.g., `'age'` if present). We'll try to automatically pick columns if found.

In [ ]:
# Identify a numeric field and a group field from the main DataFrame
main_df = dataframes[main_rs_id]

import numpy as np
numeric_field = None
group_field = None

# Try to auto-detect
for c in main_df.columns:
    try:
        # If column can be interpreted as numeric for >50% values
        vals = pd.to_numeric(main_df[c], errors='coerce')
        if vals.notnull().mean() > 0.5:
            numeric_field = c
            break
    except:
        continue

for c in main_df.columns:
    # Heuristic: If there are <10 unique non-numeric values, treat as group field
    if main_df[c].nunique() > 1 and main_df[c].nunique() <= 10:
        if not np.issubdtype(main_df[c].dtype, np.number):
            group_field = c
            break

print(f"Numeric field auto-detected: {numeric_field}")
print(f"Group field auto-detected: {group_field}")

if numeric_field is not None:
    # Ensure field is numeric
    main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
    threshold = main_df[numeric_field].median()
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (z-score) for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouped statistics
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_'+numeric_field)
        print(f"Mean {numeric_field} grouped by {group_field}:")
        display(grouped_df)
else:
    print("No numeric field found for EDA in this RecordSet.")

## 5. Visualization
Visualize the distribution of a numeric field or a relationship to a grouping variable (if possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(7,5))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=15, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(8,6))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field detected for plotting.")

## 6. Conclusion

In this notebook, we've:
- Loaded the FAIR² clinical oncology dataset via its Croissant schema using `mlcroissant`
- Explored the available record sets and fields via their unique `@id`
- Extracted all data using dynamic programming with record set IDs
- Performed basic filtering, normalization, and grouping on a numeric field
- Generated basic plots to visualize data distributions and relationships

This workflow can be applied to any Croissant-compatible dataset, using field and record set `@id` references for full reproducibility.